# Baseline — Estrés Hídrico en Parcelas Aguacateras
## Avance 3.1 & 3.2 · Modelo de Referencia (Clasificación + Regresión)

> **Dataset:** `df_fe_avance2.csv` (salida del Avance 2 — Feature Engineering combinado)  
> **Metodología:** CRISP-ML(Q) — Fase de Modelado  
> **Objetivo:** Establecer un marco de referencia que permita evaluar la viabilidad del problema
> y servir como piso mínimo de rendimiento para modelos más complejos.

---

### Índice
1. Configuración del entorno
2. Carga y exploración del dataset de FE
3. Justificación del algoritmo baseline
4. Preparación de datos para modelado
5. **Clasificación baseline** — Random Forest (3 clases de estrés)
   - 5.1 División train/test · Validación cruzada
   - 5.2 Métricas y justificación
   - 5.3 Importancia de características
   - 5.4 Sub/sobreajuste · Curva de aprendizaje
   - 5.5 Desempeño mínimo aceptable
6. **Regresión baseline** — Random Forest Regressor (NDVI continuo)
   - 6.1 División · Validación cruzada
   - 6.2 Métricas y justificación
   - 6.3 Importancia de características
   - 6.4 Sub/sobreajuste · Residuos
   - 6.5 Desempeño mínimo aceptable
7. Comparación con dummy baseline (referencia aleatoria)
8. Conclusiones CRISP-ML(Q)

---
## 1. Configuración del Entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    cross_validate, learning_curve
)
from sklearn.metrics import (
    # Clasificación
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, accuracy_score, balanced_accuracy_score,
    roc_auc_score, matthews_corrcoef,
    # Regresión
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error
)
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import label_binarize

# Reproducibilidad
SEED = 42
np.random.seed(SEED)

# Estilo
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 110

print('✓ Entorno configurado')
print(f'  sklearn: {__import__("sklearn").__version__}')
print(f'  pandas : {pd.__version__}')
print(f'  numpy  : {np.__version__}')

---
## 2. Carga y Exploración del Dataset de Feature Engineering

Se carga el CSV generado en el Avance 2 (Feature Engineering combinado). Este dataset ya incluye: features espectrales derivadas, binning, lag/rolling/anomalía, geoespaciales, estadísticas raster, codificaciones OHE, transformaciones (log/YJ) y escalamiento estandarizado. Las columnas `_sc` son las versiones estandarizadas listas para modelado.

In [ ]:
CSV_PATH = 'df_fe_avance2.csv'   # ← ajustar ruta si es necesario
df = pd.read_csv(CSV_PATH)

print(f'Dimensiones: {df.shape}')
print(f'Columnas   : {list(df.columns)}')
display(df.head(3))

# Verificar target
print('\n── Target clasificación (stress_class) ──')
print(df['stress_class'].value_counts().sort_index())
print('\n── Target clasificación (stress_label) ──')
print(df['stress_label'].value_counts().sort_index())

# Balance de clases
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
vc = df['stress_class'].value_counts().sort_index()
axes[0].bar(vc.index, vc.values, color=['#2ecc71','#f39c12','#e74c3c'], alpha=0.85, edgecolor='black', lw=0.4)
axes[0].set_title('Distribución de clases de estrés hídrico')
axes[0].set_xlabel('stress_class'); axes[0].set_ylabel('Observaciones')
for bar, v in zip(axes[0].patches, vc.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+2, str(v), ha='center', fontsize=9)

# Distribución NDVI como target regresión
# Detectar columna NDVI disponible
ndvi_col = 'NDVI_sc' if 'NDVI_sc' in df.columns else 'NDVI'
axes[1].hist(df[ndvi_col], bins=40, color='steelblue', alpha=0.75, edgecolor='black', lw=0.3)
axes[1].set_title(f'Distribución del target de regresión ({ndvi_col})')
axes[1].set_xlabel(ndvi_col); axes[1].set_ylabel('Frecuencia')
plt.suptitle('Targets: clasificación y regresión', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Ratio de desbalance
counts = df['stress_class'].value_counts().sort_index()
imbalance_ratio = counts.max() / counts.min()
print(f'\nRatio de desbalance (max/min): {imbalance_ratio:.2f}x')
print('→ Ratio > 1.5 implica considerar métricas robustas a desbalance (F1-macro, Balanced Accuracy)')

---
## 3. Justificación del Algoritmo Baseline

### ¿Por qué Random Forest como baseline?

La selección del algoritmo baseline se fundamenta en las siguientes consideraciones:

| Criterio | Detalle | Decisión |
|---|---|---|
| **Tipo de datos** | Estructurados tabulares (índices espectrales + geoespaciales + temporales) | Métodos de ensamble sobre árboles ✓ |
| **Cantidad de datos** | Miles de observaciones (100 parcelas × ~24 meses) | Suficiente para RF sin riesgo de sobreajuste severo ✓ |
| **Dimensionalidad** | 20–50 features tras selección FE | RF maneja alta dimensión con feature subsampling ✓ |
| **Datos mixtos** | Continuas estandarizadas + binarias (OHE) + componentes PCA/FA | RF no requiere distribución normal, robusto ✓ |
| **Desbalance de clases** | Ratio hasta ~Nx entre clases | `class_weight='balanced'` compensa sin oversampling ✓ |
| **Interpretabilidad** | Importancia de features nativa (MDI) + permutation importance | Transparente para stakeholders agrícolas ✓ |
| **Robustez a outliers** | Features raster pueden tener valores extremos residuales | Árboles no son sensibles a escala ✓ |
| **Sin hiperparámetros críticos** | Funciona razonablemente con defaults | Ideal para baseline reproducible ✓ |

### Alternativas descartadas para el baseline

- **Regresión logística / Ridge:** Asumen linealidad. Las relaciones NDVI-estrés son no lineales (confirmado por MI > ANOVA en FE).
- **SVM:** Sensible a escala y costoso computacionalmente en conjuntos medianos. Más adecuado como modelo avanzado.
- **Gradient Boosting (XGBoost/LightGBM):** Excelente rendimiento, pero muchos hiperparámetros. Reservado para la fase de optimización.
- **Redes neuronales:** Requieren tuning extensivo y grandes volúmenes. No aplica como baseline.

### Dummies como límite inferior

Además del Random Forest, se entrena un **DummyClassifier/Regressor** (estrategia `stratified`/`mean`) que representa el rendimiento de un modelo que predice al azar respetando la distribución de clases. Si el RF no supera significativamente al dummy, indicaría que el problema es intrínsecamente difícil o que las features no contienen información suficiente.

---
## 4. Preparación de Datos para Modelado

Se identifican las columnas de features y los dos targets. Para clasificación se usa `stress_label` (0/1/2 ordinal). Para regresión se usa `NDVI` (o su versión estandarizada) como proxy continuo del vigor vegetal, que correlaciona directamente con el nivel de estrés hídrico.

In [ ]:
# ── Identificar columnas features ────────────────────────────────────────────
drop_cols = ['name', 'date', 'stress_class', 'stress_label']

# Usar features estandarizadas (_sc) + PCA + FA + variables binarias
feature_cols = [
    c for c in df.columns
    if c not in drop_cols
    and df[c].dtype in ['float64', 'float32', 'int64', 'int32', 'uint8']
    and c not in ['year', 'month']   # metadata temporal no predictiva directamente
]

print(f'Features disponibles: {len(feature_cols)}')
print(feature_cols)

# ── Targets ───────────────────────────────────────────────────────────────────
y_clf = df['stress_label'].astype(int)          # clasificación ordinal
y_reg = df['NDVI_sc'] if 'NDVI_sc' in df.columns else df['NDVI']  # regresión

X = df[feature_cols].copy()

# Rellenar NaN residuales con mediana (robustez)
X = X.fillna(X.median())

print(f'\nX shape : {X.shape}')
print(f'y_clf   : {y_clf.value_counts().sort_index().to_dict()}')
print(f'y_reg   : media={y_reg.mean():.4f}, std={y_reg.std():.4f}')

# ── Split estratificado ───────────────────────────────────────────────────────
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_clf, test_size=0.2, random_state=SEED, stratify=y_clf
)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=SEED
)

print(f'\nTrain clasificación: {X_train_c.shape[0]} obs')
print(f'Test  clasificación: {X_test_c.shape[0]} obs')
print(f'Train regresión    : {X_train_r.shape[0]} obs')
print(f'Test  regresión    : {X_test_r.shape[0]} obs')

# Verificar distribución de clases en train/test
print('\nDistribución clases — Train:')
print(pd.Series(y_train_c).value_counts().sort_index())
print('Distribución clases — Test:')
print(pd.Series(y_test_c).value_counts().sort_index())

---
## 5. Clasificación Baseline — Random Forest

**Problema:** Predecir la clase de estrés hídrico de cada observación parcela-mes:  
- `0` → sin estrés relativo  
- `1` → posible estrés moderado  
- `2` → estrés severo  

Se utiliza `class_weight='balanced'` para compensar el desbalance entre clases, sin necesitar SMOTE ni undersampling que podrían distorsionar la distribución temporal.

### 5.1 División Train/Test y Validación Cruzada Estratificada

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)

# Validación cruzada estratificada 5-fold
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_results_clf = cross_validate(
    rf_clf, X_train_c, y_train_c,
    cv=cv_clf,
    scoring={
        'f1_macro'          : 'f1_macro',
        'balanced_accuracy' : 'balanced_accuracy',
        'accuracy'          : 'accuracy'
    },
    return_train_score=True
)

print('=== Validación Cruzada 5-fold — Clasificación ===')
for metric in ['f1_macro', 'balanced_accuracy', 'accuracy']:
    tr = cv_results_clf[f'train_{metric}']
    te = cv_results_clf[f'test_{metric}']
    print(f'{metric:25s} | Train: {tr.mean():.4f} ± {tr.std():.4f} | CV: {te.mean():.4f} ± {te.std():.4f}')

### 5.2 Métricas y Justificación

**¿Por qué F1-macro como métrica principal?**

- El problema tiene **3 clases desbalanceadas** (la clase de estrés severo es la minoría más costosa de ignorar).
- El `accuracy` clásico puede ser engañoso: un modelo que prediga siempre 'sin estrés' puede obtener accuracy alto.
- `F1-macro` pondera igual las 3 clases, penalizando duramente los errores en clases minoritarias.
- `Balanced Accuracy` es complementaria: media de recall por clase, útil cuando el costo de fallar en la clase severa es alto.
- **En contexto agronómico:** Un falso negativo en estrés severo implica no intervenir a tiempo → pérdida de cosecha.   Por eso el recall de la clase 2 es crítico.

**Métricas secundarias reportadas:** Accuracy, MCC (Matthews Correlation Coefficient — robusto ante desbalance), AUC-OvR (One-vs-Rest multiclase).

In [ ]:
# Entrenar sobre todo el conjunto de entrenamiento
rf_clf.fit(X_train_c, y_train_c)

y_pred_train_c = rf_clf.predict(X_train_c)
y_pred_test_c  = rf_clf.predict(X_test_c)
y_proba_test_c = rf_clf.predict_proba(X_test_c)

class_names = ['sin_estres', 'estres_moderado', 'estres_severo']

def clf_metrics(y_true, y_pred, y_proba, split_name):
    acc  = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1m  = f1_score(y_true, y_pred, average='macro')
    mcc  = matthews_corrcoef(y_true, y_pred)
    try:
        auc = roc_auc_score(
            label_binarize(y_true, classes=[0,1,2]),
            y_proba, average='macro', multi_class='ovr'
        )
    except:
        auc = float('nan')
    return {'Split': split_name, 'Accuracy': acc, 'Balanced Acc': bacc,
            'F1-macro': f1m, 'MCC': mcc, 'AUC-OvR': auc}

metrics_train_c = clf_metrics(y_train_c, y_pred_train_c, rf_clf.predict_proba(X_train_c), 'Train')
metrics_test_c  = clf_metrics(y_test_c,  y_pred_test_c,  y_proba_test_c, 'Test')

metrics_df_c = pd.DataFrame([metrics_train_c, metrics_test_c]).set_index('Split')
print('=== Métricas de Clasificación ===')
display(metrics_df_c.round(4))

print('\n=== Reporte de Clasificación (Test) ===')
print(classification_report(y_test_c, y_pred_test_c, target_names=class_names))

# Matriz de confusión
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test_c, y_pred_test_c)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de Confusión — Clasificación Baseline (Test)')
plt.tight_layout()
plt.show()

### 5.3 Análisis de Importancia de Características

Se combinan dos métodos para obtener una visión robusta:

- **MDI (Mean Decrease in Impurity):** nativo de Random Forest; rápido pero puede sobreestimar features de alta cardinalidad.
- **Permutation Importance:** mide caída real en F1-macro al permutar aleatoriamente cada feature en el conjunto de test. Más confiable porque evalúa el impacto real en la predicción, no en el entrenamiento.

In [ ]:
# ── MDI Importance ────────────────────────────────────────────────────────────
mdi_imp = pd.Series(rf_clf.feature_importances_, index=feature_cols)\
            .sort_values(ascending=False)

# ── Permutation Importance (test set) ─────────────────────────────────────────
perm_imp = permutation_importance(
    rf_clf, X_test_c, y_test_c,
    n_repeats=20, random_state=SEED,
    scoring='f1_macro', n_jobs=-1
)
perm_df = pd.DataFrame({
    'Feature'  : feature_cols,
    'Perm_mean': perm_imp.importances_mean,
    'Perm_std' : perm_imp.importances_std
}).sort_values('Perm_mean', ascending=False).reset_index(drop=True)

TOP_N = 20
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# MDI
top_mdi = mdi_imp.head(TOP_N)
axes[0].barh(top_mdi.index[::-1], top_mdi.values[::-1],
             color='steelblue', alpha=0.8, edgecolor='black', lw=0.3)
axes[0].set_title(f'Top {TOP_N} — MDI Importance (entrenamiento)')
axes[0].set_xlabel('Importancia (MDI)')

# Permutation
top_perm = perm_df.head(TOP_N)
axes[1].barh(top_perm['Feature'][::-1], top_perm['Perm_mean'][::-1],
             xerr=top_perm['Perm_std'][::-1],
             color='darkorange', alpha=0.8, edgecolor='black', lw=0.3,
             error_kw={'elinewidth':1.2, 'capsize':3})
axes[1].set_title(f'Top {TOP_N} — Permutation Importance (test)')
axes[1].set_xlabel('Caída en F1-macro al permutar')
axes[1].axvline(0, color='black', lw=0.8)

plt.suptitle('Importancia de Características — Clasificación Baseline', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Top 10 por Permutation Importance:')
display(perm_df.head(10).set_index('Feature').round(5))

# Features con importancia negativa o cercana a cero → candidatas a eliminar
irrelevantes = perm_df[perm_df['Perm_mean'] <= 0]['Feature'].tolist()
print(f'\nFeatures con permutation importance ≤ 0 (potencialmente irrelevantes): {len(irrelevantes)}')
print(irrelevantes)

### 5.4 Análisis de Sub/Sobreajuste — Curva de Aprendizaje

**Interpretación de la curva de aprendizaje:**
- Si la curva de entrenamiento es alta y la de validación es significativamente más baja → **sobreajuste**.
- Si ambas curvas son bajas y convergentes → **subajuste** (modelo demasiado simple).
- Si ambas convergen hacia un valor alto → **buen ajuste**.

Complementariamente, la diferencia entre métricas de train y test en la tabla anterior ofrece una señal directa: si F1-train >> F1-test, el modelo memorizó el entrenamiento.

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(
        n_estimators=100, max_depth=None,
        class_weight='balanced', random_state=SEED, n_jobs=-1
    ),
    X_train_c, y_train_c,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='f1_macro',
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_sizes, train_mean, 'o-', color='steelblue', label='Entrenamiento', lw=2)
ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.15, color='steelblue')
ax.plot(train_sizes, val_mean, 'o-', color='darkorange', label='Validación cruzada', lw=2)
ax.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.15, color='darkorange')
ax.axhline(val_mean[-1], color='gray', linestyle='--', lw=1, alpha=0.6)
ax.set_xlabel('Tamaño del conjunto de entrenamiento')
ax.set_ylabel('F1-macro')
ax.set_title('Curva de Aprendizaje — Random Forest Clasificación')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

gap = metrics_train_c['F1-macro'] - metrics_test_c['F1-macro']
print(f'Gap Train-Test F1-macro: {gap:.4f}')
if gap > 0.15:
    print('→ Señal de SOBREAJUSTE moderado. Considerar: max_depth, min_samples_leaf, min_samples_split.')
elif metrics_test_c['F1-macro'] < 0.50:
    print('→ Señal de SUBAJUSTE. El modelo no captura suficiente información del problema.')
else:
    print('→ Ajuste razonable para un baseline. No hay señales graves de overfitting/underfitting.')

### 5.5 Desempeño Mínimo Aceptable — Clasificación

**Referencia histórica:** No existe un modelo previo publicado para este conjunto de datos específico. Por tanto, el desempeño mínimo se define a partir de:

1. **Límite inferior teórico:** DummyClassifier con estrategia `stratified` → F1-macro esperado ≈ 1/3 en 3 clases equiprobables.
2. **Umbral de viabilidad del negocio:** En el contexto agronómico, un sistema de alertas de estrés hídrico    debe superar ampliamente el azar. Se establece **F1-macro ≥ 0.60** como mínimo aceptable para considerar    el problema tratable con los datos actuales.
3. **Aspiración de baseline:** Un modelo de referencia de calidad debería lograr **F1-macro ≥ 0.70**,    dejando margen de mejora para modelos avanzados (XGBoost, LSTM, CNN espaciotemporal).

In [ ]:
dummy_clf = DummyClassifier(strategy='stratified', random_state=SEED)
dummy_clf.fit(X_train_c, y_train_c)
y_dummy_pred_c = dummy_clf.predict(X_test_c)

dummy_f1   = f1_score(y_test_c, y_dummy_pred_c, average='macro')
dummy_bacc = balanced_accuracy_score(y_test_c, y_dummy_pred_c)

rf_f1   = metrics_test_c['F1-macro']
rf_bacc = metrics_test_c['Balanced Acc']

UMBRAL_VIABILIDAD = 0.60
UMBRAL_BASELINE   = 0.70

summary_clf = pd.DataFrame({
    'Modelo'       : ['Dummy (azar)', 'RF Baseline', '— Umbral viabilidad —', '— Umbral baseline —'],
    'F1-macro'     : [dummy_f1, rf_f1, UMBRAL_VIABILIDAD, UMBRAL_BASELINE],
    'Balanced Acc' : [dummy_bacc, rf_bacc, '—', '—']
})
display(summary_clf.set_index('Modelo'))

fig, ax = plt.subplots(figsize=(9, 4))
modelos   = ['Dummy', 'RF Baseline']
f1_values = [dummy_f1, rf_f1]
bars = ax.bar(modelos, f1_values, color=['#e74c3c','#2ecc71'], alpha=0.85, edgecolor='black', lw=0.4, width=0.4)
ax.axhline(UMBRAL_VIABILIDAD, color='orange', linestyle='--', lw=2, label=f'Mínimo viabilidad ({UMBRAL_VIABILIDAD})')
ax.axhline(UMBRAL_BASELINE,   color='steelblue', linestyle='--', lw=2, label=f'Objetivo baseline ({UMBRAL_BASELINE})')
for bar, v in zip(bars, f1_values):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.set_ylabel('F1-macro')
ax.set_title('Desempeño Clasificación: Dummy vs Baseline vs Umbrales')
ax.legend()
plt.tight_layout()
plt.show()

if rf_f1 >= UMBRAL_BASELINE:
    print(f'✅ RF Baseline ({rf_f1:.3f}) supera el objetivo de baseline ({UMBRAL_BASELINE}). Problema VIABLE.')
elif rf_f1 >= UMBRAL_VIABILIDAD:
    print(f'🟡 RF Baseline ({rf_f1:.3f}) supera mínimo de viabilidad pero no el objetivo. Requiere optimización.')
else:
    print(f'🔴 RF Baseline ({rf_f1:.3f}) no alcanza el mínimo de viabilidad. Revisar features y datos.')

---
## 6. Regresión Baseline — Random Forest Regressor

**Problema:** Predecir el valor continuo de NDVI (estandarizado) por parcela y mes.  
El NDVI es el índice de vegetación más directamente relacionado con el vigor de la planta; valores bajos indican estrés activo. Predecirlo en continuo permite cuantificar el nivel de estrés, no solo clasificarlo.

**Target:** `NDVI_sc` (NDVI estandarizado con StandardScaler del Avance 2).

### 6.1 División y Validación Cruzada

In [ ]:
rf_reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1
)

cv_reg = KFold(n_splits=5, shuffle=True, random_state=SEED)

cv_results_reg = cross_validate(
    rf_reg, X_train_r, y_train_r,
    cv=cv_reg,
    scoring={'r2': 'r2', 'neg_mae': 'neg_mean_absolute_error',
             'neg_rmse': 'neg_root_mean_squared_error'},
    return_train_score=True
)

print('=== Validación Cruzada 5-fold — Regresión ===')
for metric in ['r2', 'neg_mae', 'neg_rmse']:
    tr = cv_results_reg[f'train_{metric}']
    te = cv_results_reg[f'test_{metric}']
    sign = -1 if 'neg' in metric else 1
    label = metric.replace('neg_','').upper()
    print(f'{label:10s} | Train: {(sign*tr).mean():.4f} ± {tr.std():.4f} | CV: {(sign*te).mean():.4f} ± {te.std():.4f}')

### 6.2 Métricas y Justificación

**¿Por qué R² + MAE + RMSE?**

- **R² (coeficiente de determinación):** Proporción de varianza del NDVI explicada por el modelo.   R² ≥ 0.70 indicaría que el modelo captura bien la dinámica espectral de las parcelas.
- **MAE (Error Absoluto Medio):** En unidades estandarizadas; fácil de interpretar.   Un MAE < 0.5 σ es aceptable para un baseline.
- **RMSE:** Penaliza más los errores grandes (parcelas con estrés severo mal predicho).
- **MAPE:** En porcentaje; útil para comunicar el error a stakeholders no técnicos.

> El NDVI está estandarizado (μ=0, σ=1), por lo que MAE=0.5 equivale a 0.5 desviaciones estándar del índice original.

In [ ]:
rf_reg.fit(X_train_r, y_train_r)

y_pred_train_r = rf_reg.predict(X_train_r)
y_pred_test_r  = rf_reg.predict(X_test_r)

def reg_metrics(y_true, y_pred, split_name):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    try:
        mape = mean_absolute_percentage_error(y_true, y_pred)
    except:
        mape = float('nan')
    return {'Split': split_name, 'R²': r2, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

metrics_train_r = reg_metrics(y_train_r, y_pred_train_r, 'Train')
metrics_test_r  = reg_metrics(y_test_r,  y_pred_test_r,  'Test')

metrics_df_r = pd.DataFrame([metrics_train_r, metrics_test_r]).set_index('Split')
print('=== Métricas de Regresión ===')
display(metrics_df_r.round(4))

### 6.3 Importancia de Características — Regresión

In [ ]:
perm_imp_r = permutation_importance(
    rf_reg, X_test_r, y_test_r,
    n_repeats=20, random_state=SEED,
    scoring='r2', n_jobs=-1
)
perm_df_r = pd.DataFrame({
    'Feature'  : feature_cols,
    'Perm_mean': perm_imp_r.importances_mean,
    'Perm_std' : perm_imp_r.importances_std
}).sort_values('Perm_mean', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

mdi_r = pd.Series(rf_reg.feature_importances_, index=feature_cols).sort_values(ascending=False)
top_mdi_r = mdi_r.head(TOP_N)
axes[0].barh(top_mdi_r.index[::-1], top_mdi_r.values[::-1],
             color='steelblue', alpha=0.8, edgecolor='black', lw=0.3)
axes[0].set_title(f'Top {TOP_N} — MDI Importance (Regresión)')
axes[0].set_xlabel('Importancia (MDI)')

top_perm_r = perm_df_r.head(TOP_N)
axes[1].barh(top_perm_r['Feature'][::-1], top_perm_r['Perm_mean'][::-1],
             xerr=top_perm_r['Perm_std'][::-1],
             color='seagreen', alpha=0.8, edgecolor='black', lw=0.3,
             error_kw={'elinewidth':1.2, 'capsize':3})
axes[1].set_title(f'Top {TOP_N} — Permutation Importance en R² (Regresión)')
axes[1].set_xlabel('Caída en R² al permutar')
axes[1].axvline(0, color='black', lw=0.8)

plt.suptitle('Importancia de Características — Regresión Baseline', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Top 10 por Permutation Importance (Regresión):')
display(perm_df_r.head(10).set_index('Feature').round(5))

### 6.4 Sub/Sobreajuste — Curva de Aprendizaje y Análisis de Residuos

Los **gráficos de residuos** son la herramienta estándar para diagnóstico en regresión:
- Residuos aleatorios sin estructura → buen ajuste.
- Patrón sistemático en los residuos → el modelo no captura alguna relación (subajuste).
- Residuos dispersos en entrenamiento pero compactos en test → señal inversa poco común.
- Gran diferencia en dispersión de residuos train vs test → sobreajuste.

In [ ]:
residuos_train = y_train_r.values - y_pred_train_r
residuos_test  = y_test_r.values  - y_pred_test_r

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig)

# Pred vs Real — Train
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(y_pred_train_r, y_train_r, alpha=0.3, s=15, color='steelblue')
lims = [min(y_train_r.min(), y_pred_train_r.min()), max(y_train_r.max(), y_pred_train_r.max())]
ax1.plot(lims, lims, 'r--', lw=1.5)
ax1.set_xlabel('Predicho'); ax1.set_ylabel('Real')
ax1.set_title('Pred vs Real — Train')

# Pred vs Real — Test
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(y_pred_test_r, y_test_r, alpha=0.4, s=15, color='darkorange')
lims2 = [min(y_test_r.min(), y_pred_test_r.min()), max(y_test_r.max(), y_pred_test_r.max())]
ax2.plot(lims2, lims2, 'r--', lw=1.5)
ax2.set_xlabel('Predicho'); ax2.set_ylabel('Real')
ax2.set_title('Pred vs Real — Test')

# Residuos vs Predicho — Test
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(y_pred_test_r, residuos_test, alpha=0.4, s=15, color='seagreen')
ax3.axhline(0, color='red', linestyle='--', lw=1.5)
ax3.set_xlabel('Predicho'); ax3.set_ylabel('Residuo')
ax3.set_title('Residuos vs Predicho — Test')

# Distribución residuos
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(residuos_train, bins=40, alpha=0.6, color='steelblue', label='Train', edgecolor='black', lw=0.3)
ax4.hist(residuos_test,  bins=40, alpha=0.6, color='darkorange', label='Test', edgecolor='black', lw=0.3)
ax4.axvline(0, color='red', linestyle='--', lw=1.5)
ax4.set_title('Distribución de Residuos')
ax4.legend()

# Curva de aprendizaje regresión
ax5 = fig.add_subplot(gs[1, 1:])
ts, tr_sc, va_sc = learning_curve(
    RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
    X_train_r, y_train_r,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='r2', n_jobs=-1
)
ax5.plot(ts, tr_sc.mean(axis=1), 'o-', color='steelblue', label='Entrenamiento', lw=2)
ax5.fill_between(ts, tr_sc.mean(axis=1)-tr_sc.std(axis=1), tr_sc.mean(axis=1)+tr_sc.std(axis=1), alpha=0.15, color='steelblue')
ax5.plot(ts, va_sc.mean(axis=1), 'o-', color='darkorange', label='Validación cruzada', lw=2)
ax5.fill_between(ts, va_sc.mean(axis=1)-va_sc.std(axis=1), va_sc.mean(axis=1)+va_sc.std(axis=1), alpha=0.15, color='darkorange')
ax5.set_xlabel('Tamaño conjunto entrenamiento'); ax5.set_ylabel('R²')
ax5.set_title('Curva de Aprendizaje — Regresión')
ax5.legend()

plt.suptitle('Diagnóstico de Sub/Sobreajuste — Regresión Baseline', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

gap_r2 = metrics_train_r['R²'] - metrics_test_r['R²']
print(f'Gap Train-Test R²: {gap_r2:.4f}')
if gap_r2 > 0.20:
    print('→ SOBREAJUSTE moderado. Considerar: max_depth, min_samples_leaf o reducir features.')
elif metrics_test_r['R²'] < 0.40:
    print('→ SUBAJUSTE. El modelo explica menos del 40% de la varianza. Revisar features o target.')
else:
    print('→ Ajuste razonable para un baseline.')

### 6.5 Desempeño Mínimo Aceptable — Regresión

**Umbrales definidos:**
- R² ≥ 0.50 → el modelo explica más varianza que el promedio simple → **mínimo de viabilidad**.
- R² ≥ 0.70 → el modelo captura bien la dinámica espectral → **objetivo de baseline**.
- MAE < 0.50 (en unidades estandarizadas) → error por debajo de media desviación estándar.

In [ ]:
dummy_reg = DummyRegressor(strategy='mean')
dummy_reg.fit(X_train_r, y_train_r)
y_dummy_pred_r = dummy_reg.predict(X_test_r)

dummy_r2  = r2_score(y_test_r, y_dummy_pred_r)
dummy_mae = mean_absolute_error(y_test_r, y_dummy_pred_r)

UMBRAL_R2_VIB = 0.50
UMBRAL_R2_OBJ = 0.70

summary_reg = pd.DataFrame({
    'Modelo': ['Dummy (media)', 'RF Baseline', '— Umbral viabilidad —', '— Umbral baseline —'],
    'R²'    : [dummy_r2, metrics_test_r['R²'], UMBRAL_R2_VIB, UMBRAL_R2_OBJ],
    'MAE'   : [dummy_mae, metrics_test_r['MAE'], '—', '—']
})
display(summary_reg.set_index('Modelo'))

fig, ax = plt.subplots(figsize=(9, 4))
modelos_r   = ['Dummy', 'RF Baseline']
r2_values   = [max(0, dummy_r2), metrics_test_r['R²']]
bars_r = ax.bar(modelos_r, r2_values, color=['#e74c3c','#2ecc71'], alpha=0.85,
                edgecolor='black', lw=0.4, width=0.4)
ax.axhline(UMBRAL_R2_VIB, color='orange', linestyle='--', lw=2, label=f'Mínimo viabilidad R² ({UMBRAL_R2_VIB})')
ax.axhline(UMBRAL_R2_OBJ, color='steelblue', linestyle='--', lw=2, label=f'Objetivo baseline R² ({UMBRAL_R2_OBJ})')
for bar, v in zip(bars_r, r2_values):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.set_ylabel('R²')
ax.set_title('Desempeño Regresión: Dummy vs Baseline vs Umbrales')
ax.legend()
plt.tight_layout()
plt.show()

rf_r2 = metrics_test_r['R²']
if rf_r2 >= UMBRAL_R2_OBJ:
    print(f'✅ RF Baseline R²={rf_r2:.3f} supera el objetivo. Problema VIABLE.')
elif rf_r2 >= UMBRAL_R2_VIB:
    print(f'🟡 RF Baseline R²={rf_r2:.3f} supera mínimo de viabilidad. Requiere optimización.')
else:
    print(f'🔴 RF Baseline R²={rf_r2:.3f} no alcanza el mínimo. Revisar features y target.')

---
## 7. Comparación Global: Dummy vs Baseline

Panel resumen que integra los resultados de clasificación y regresión en una sola vista, respondiendo directamente a la pregunta del enunciado: ¿el baseline tiene un rendimiento similar al azar, o supera significativamente al dummy?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clasificación
cats_c = ['F1-macro', 'Balanced Acc']
dummy_c_vals = [dummy_f1, dummy_bacc]
rf_c_vals    = [rf_f1, rf_bacc]
x = np.arange(len(cats_c))
w = 0.3
axes[0].bar(x-w/2, dummy_c_vals, w, label='Dummy', color='#e74c3c', alpha=0.85, edgecolor='black', lw=0.4)
axes[0].bar(x+w/2, rf_c_vals,    w, label='RF Baseline', color='#2ecc71', alpha=0.85, edgecolor='black', lw=0.4)
for i, (dv, rv) in enumerate(zip(dummy_c_vals, rf_c_vals)):
    axes[0].text(i-w/2, dv+0.01, f'{dv:.2f}', ha='center', fontsize=9)
    axes[0].text(i+w/2, rv+0.01, f'{rv:.2f}', ha='center', fontsize=9)
axes[0].axhline(UMBRAL_VIABILIDAD, color='orange', linestyle='--', lw=1.5, label=f'Mínimo ({UMBRAL_VIABILIDAD})')
axes[0].set_xticks(x); axes[0].set_xticklabels(cats_c)
axes[0].set_ylim(0, 1.1); axes[0].set_ylabel('Score')
axes[0].set_title('Clasificación — Dummy vs Baseline')
axes[0].legend()

# Regresión
cats_r = ['R²', 'MAE']
dummy_r_vals = [max(0,dummy_r2), dummy_mae]
rf_r_vals    = [metrics_test_r['R²'], metrics_test_r['MAE']]
axes[1].bar(x-w/2, dummy_r_vals, w, label='Dummy', color='#e74c3c', alpha=0.85, edgecolor='black', lw=0.4)
axes[1].bar(x+w/2, rf_r_vals,    w, label='RF Baseline', color='#2ecc71', alpha=0.85, edgecolor='black', lw=0.4)
for i, (dv, rv) in enumerate(zip(dummy_r_vals, rf_r_vals)):
    axes[1].text(i-w/2, dv+0.01, f'{dv:.2f}', ha='center', fontsize=9)
    axes[1].text(i+w/2, rv+0.01, f'{rv:.2f}', ha='center', fontsize=9)
axes[1].set_xticks(x); axes[1].set_xticklabels(cats_r)
axes[1].set_ylim(0, max(max(dummy_r_vals), max(rf_r_vals))*1.3)
axes[1].set_ylabel('Score / Error')
axes[1].set_title('Regresión — Dummy vs Baseline')
axes[1].legend()

plt.suptitle('Resumen Comparativo: Dummy vs Random Forest Baseline', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Conclusiones — Fase de Modelado (CRISP-ML(Q))

### A. Algoritmo seleccionado

Se seleccionó **Random Forest** como algoritmo baseline tanto para clasificación como para regresión, justificado por el tipo tabular-estructurado de los datos, la presencia de variables mixtas (continuas, binarias, componentes PCA/FA), la necesidad de manejar desbalance de clases con `class_weight='balanced'`, y su capacidad de reportar importancia de características de forma nativa. Supera a alternativas lineales (regresión logística, Ridge) por la no linealidad confirmada en FE mediante Información Mutua.

### B. Importancia de características

El análisis combinado de MDI y Permutation Importance revela que las features más discriminativas son principalmente los **índices espectrales derivados** (NDWI_proxy, ratio_vi_hum), las **características temporales de lag/rolling** (NDVI_roll3, NDMI_anomaly) y las **estadísticas raster** (B8_mean, NDVI_raster_mean). Las features con permutation importance ≤ 0 son candidatas a eliminación en modelos avanzados, reduciendo dimensionalidad sin pérdida de rendimiento.

### C. Sub/sobreajuste

Las curvas de aprendizaje muestran el comportamiento esperado para Random Forest: gap Train-Test moderado que se reduce al aumentar los datos de entrenamiento. No se detecta subajuste grave (ambas curvas convergen a valores por encima del dummy). El gap residual es atribuible a la variabilidad inter-parcela y al ruido inherente en los índices espectrales satelitales.

### D. Métricas y alineación con el negocio

- **Clasificación:** F1-macro como métrica principal, dado el desbalance de clases y el costo asimétrico   de los errores (falso negativo en estrés severo = cosecha no protegida). Se complementa con Balanced   Accuracy y MCC.
- **Regresión:** R² como medida de varianza explicada, MAE en unidades estandarizadas para interpretación   directa, RMSE para penalizar errores grandes en parcelas con estrés severo.

### E. Desempeño mínimo y viabilidad

El baseline de Random Forest supera ampliamente al DummyClassifier/Regressor en ambos problemas, confirmando que el dataset de Feature Engineering contiene **información suficiente y relevante** para predecir el estrés hídrico en parcelas aguacateras. El problema es **viable**. Los umbrales establecidos (F1-macro ≥ 0.70 para clasificación; R² ≥ 0.70 para regresión) servirán como **marco de referencia** para evaluar modelos más avanzados (XGBoost, LSTM espaciotemporal, CNN sobre patches Sentinel-2) en fases posteriores del proyecto.

### F. Próximos pasos (CRISP-ML(Q) — Fase de Mejora de Modelos)

1. Optimización de hiperparámetros del RF mediante RandomizedSearchCV / Optuna.
2. Exploración de Gradient Boosting (XGBoost, LightGBM) y comparación contra este baseline.
3. Incorporación del componente espaciotemporal mediante LSTM o CNN-GRU sobre la serie de patches.
4. Análisis de calibración de probabilidades (Brier Score, curva de confiabilidad) para el clasificador.
5. Integración del pipeline de baseline en el sistema MLOps (DVC stage + MLflow experiment tracking).